# Deployment and graduation

## Learning goals
- Move from a Mac client to a Linux serving host
- Read a real multi-stage launch end to end
- Leave with a checklist for going to production

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## From Mac to Linux

You built every mental model on your Mac with simulations. Production is the same graph on real hardware: install vLLM-Omni on a Linux + NVIDIA host, launch the stages, and point your client at the HTTP server. Your Mac becomes the *client*.

In [ ]:
from dataclasses import dataclass
@dataclass
class Deployment:
    model: str
    version: str
    port: int = 8091
    def commands(self):
        return [
            f'uv pip install vllm=={self.version} --torch-backend=auto',
            f'git clone --branch v{self.version} https://github.com/vllm-project/vllm-omni.git',
            f'vllm serve {self.model} --omni --port {self.port}',
        ]
d = Deployment('Qwen/Qwen3-Omni-30B-A3B-Instruct', '0.24.1')
for c in d.commands():
    print('$', c)

## Production checklist

- pin the model + vLLM-Omni version
- scale the **bottleneck stage** first (the Talker for speech; the DiT for images)
- keep stages on one node while payloads are large (shared-memory connector); go multi-node with Mooncake only when you must
- benchmark with warmup dropped and the tail (p99) reported
- watch per-stage capacity, not just aggregate throughput

In [ ]:
checklist = ['pin versions', 'scale bottleneck', 'right connector',
             'warmup + p99', 'per-stage metrics']
for i, item in enumerate(checklist, 1):
    print(f'{i}. {item}')

## Exercise

You are deploying Qwen3-Omni for live voice. Which stage do you give the most replicas, and which connector do you start with? Justify from what you learned in nb07 and nb08.

In [ ]:
# Solution
print('Most replicas -> Talker (it decodes ~3.6x more tokens; it is the bottleneck).')
print('Connector -> start single-node shared-memory (5.49/0.53 ms); go Mooncake only if you')
print('must span nodes for accelerators.')

## Graduation

You can now read the vLLM-Omni architecture, reason about its performance, and deploy it. Go build something. Primary sources: the [docs](https://docs.vllm.ai/projects/vllm-omni/en/latest/), the [repo](https://github.com/vllm-project/vllm-omni), and the [paper](https://arxiv.org/abs/2602.02204).